# 21 — Recursive RAG

> **Tier 4 | Agentic & Self-Improving**

## What is Recursive RAG?

Standard RAG retrieves a fixed set of chunks at one granularity. **Recursive RAG**
(also called *Small-to-Big* or *Parent Document Retrieval*) indexes the same
content at **two levels**:

- **Fine chunks** — individual paragraphs/list-items/tables (~150–300 chars)
- **Coarse chunks** — entire sections (title + all child elements, ~600–1500 chars)

Retrieval starts at the **fine level** for precision. If a retrieved chunk's
similarity score falls below a confidence threshold, the system **recurses up**
to its parent coarse chunk — trading precision for broader context. High-confidence
hits stay fine; low-confidence hits get more context automatically.

### The recursion rule

```
score ≥ CONFIDENCE_THRESH  →  use fine chunk  (precise, targeted)
score <  CONFIDENCE_THRESH  →  replace with parent coarse chunk  (more context)
```

## PDF Framework: unstructured

This notebook introduces **unstructured** — a document intelligence library that
classifies PDF content into semantic element types before chunking.

| Element type | Meaning |
|---|---|
| `Title` | Section heading → hierarchy boundary |
| `NarrativeText` | Body paragraph → fine chunk |
| `ListItem` | Bullet / numbered item → fine chunk |
| `Table` | Table content → fine chunk (kept whole) |
| `FigureCaption` | Caption text → fine chunk |
| `Header` / `Footer` | Page chrome → filtered out |

We use `strategy="fast"` (pdfminer.six under the hood — no OCR, no GPU) and
build a parent-child map from the element stream.

## Flow Diagram

```mermaid
flowchart TD
    subgraph INDEX ["🔵  INDEXING — unstructured"]
        PDF(["📄 climate.pdf"])
        PDF --> PART["partition_pdf(strategy=fast)\nelement classification"]
        PART --> HIER["Build hierarchy:\nTitle → [NarrativeText, ListItem, Table]"]
        HIER --> FINE["Fine chunks\n(individual elements)\nlevel='fine'"]
        HIER --> COARSE["Coarse chunks\n(Title + all children)\nlevel='coarse'"]
        FINE --> EMBA["Embed — Titan V2"]
        COARSE --> EMBB["Embed — Titan V2"]
        EMBA --> QDRANT[("Qdrant\nrecursive_rag_21")]
        EMBB --> QDRANT
    end

    subgraph RECURSE ["🔄  RECURSIVE RETRIEVAL"]
        Q(["❓ Query"])
        Q --> RFINE["Retrieve fine chunks\n(top-K, filter: level=fine)"]
        QDRANT --> RFINE
        RFINE --> CONF{{"score ≥\nCONF_THRESH?"}}
        CONF -->|"Yes — high confidence"| KEEP["Keep fine chunk\n(precise context)"]
        CONF -->|"No — low confidence"| PARENT["Recurse up →\nparent coarse chunk\n(section-level context)"]
        KEEP --> MERGE["Merge + dedup"]
        PARENT --> MERGE
        MERGE --> GEN["Generate answer"]
        GEN --> FINAL(["✅ Final answer"])
    end

    style INDEX  fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style RECURSE fill:#f0fdf4,stroke:#16a34a,color:#14532d
```


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — unstructured"]
        PDF(["📄 climate.pdf"])
        PDF --> PART["partition_pdf(strategy=fast)\nelement classification"]
        PART --> HIER["Build hierarchy:\nTitle → [NarrativeText, ListItem, Table]"]
        HIER --> FINE["Fine chunks\n(individual elements)\nlevel='fine'"]
        HIER --> COARSE["Coarse chunks\n(Title + all children)\nlevel='coarse'"]
        FINE --> EMBA["Embed — Titan V2"]
        COARSE --> EMBB["Embed — Titan V2"]
        EMBA --> QDRANT[("Qdrant\nrecursive_rag_21")]
        EMBB --> QDRANT
    end

    subgraph RECURSE ["🔄  RECURSIVE RETRIEVAL"]
        Q(["❓ Query"])
        Q --> RFINE["Retrieve fine chunks\n(top-K, filter: level=fine)"]
        QDRANT --> RFINE
        RFINE --> CONF{{"score ≥\nCONF_THRESH?"}}
        CONF -->|"Yes — high confidence"| KEEP["Keep fine chunk\n(precise context)"]
        CONF -->|"No — low confidence"| PARENT["Recurse up →\nparent coarse chunk\n(section-level context)"]
        KEEP --> MERGE["Merge + dedup"]
        PARENT --> MERGE
        MERGE --> GEN["Generate answer"]
        GEN --> FINAL(["✅ Final answer"])
    end

    style INDEX  fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style RECURSE fill:#f0fdf4,stroke:#16a34a,color:#14532d
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = [
    "boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
    "strands-agents", "pdfminer.six", "python-dotenv",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, re
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv

import boto3
from pdfminer.high_level import extract_pages
from pdfminer.layout import LAParams, LTTextBox, LTChar
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
)

# ── Lightweight element-type stubs (mirror unstructured's API surface) ──
# We classify each LTTextBox using heuristics so the rest of the notebook
# code (KEEP_TYPES, SKIP_TYPES, isinstance checks) works identically.

class _El:
    def __init__(self, text, page, metadata=None):
        self.text = text
        class _Meta: pass
        self.metadata       = _Meta()
        self.metadata.page_number = page

class Title(_El):        pass
class NarrativeText(_El): pass
class ListItem(_El):     pass
class Table(_El):        pass
class FigureCaption(_El): pass
class Header(_El):       pass
class Footer(_El):       pass
class PageBreak(_El):
    def __init__(self): self.text=''; self.metadata=None
class Image(_El):        pass

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK  (unstructured replaced with pdfminer.six classifier)")


## Step 2 — Configuration

In [ ]:
# AWS / Bedrock
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

# Vector DB
QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

# Collection
COLLECTION_NAME = "recursive_rag_21"
EMBEDDING_DIM   = 1024

# Retrieval
TOP_K             = 6      # fine-level candidates
CONFIDENCE_THRESH = 0.75   # below this → recurse up to parent coarse chunk

# Fine/Coarse sizes
MAX_FINE_CHARS   = 400     # element text kept as-is if below this
MAX_COARSE_CHARS = 1500    # section text truncated at this limit

# Element types to keep (filter out page chrome)
KEEP_TYPES = (Title, NarrativeText, ListItem, Table, FigureCaption)
SKIP_TYPES = (Header, Footer, PageBreak, Image)

# PDF
PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection       : {COLLECTION_NAME}")
print(f"PDF              : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Confidence thresh: {CONFIDENCE_THRESH}")
print(f"TOP_K (fine)     : {TOP_K}")


## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name     = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5,
               level_filter: Optional[str] = None) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            filt = None
            if level_filter:
                filt = Filter(must=[
                    FieldCondition(key='metadata.level', match=MatchValue(value=level_filter))
                ])
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k,
                with_payload=True, query_filter=filt)
            return [{'text'    : p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score'   : p.score,
                     'id'      : str(p.id)}
                    for p in resp.points]

    def get_by_section_id(self, section_id: str) -> Optional[Dict]:
        """Fetch the coarse chunk for a given section_id."""
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            filt = Filter(must=[
                FieldCondition(key='metadata.level',      match=MatchValue(value='coarse')),
                FieldCondition(key='metadata.section_id', match=MatchValue(value=section_id)),
            ])
            resp = self._qdrant.query_points(
                collection_name=self.name, query=[0.0] * EMBEDDING_DIM,
                limit=1, with_payload=True, query_filter=filt)
            if resp.points:
                p = resp.points[0]
                return {'text': p.payload.get('text',''), 'metadata': p.payload.get('metadata',{})}
        return None

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0

print("VectorStore defined.")


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    return str(Agent(model=_model, system_prompt=system)(prompt))

test_emb = embed_text("recursive rag unstructured test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Strands BedrockModel ready.")


## Step 5 — PDF Loading with pdfminer.six (element classifier)

`unstructured` wraps pdfminer.six but requires system-level OCR tools on Windows.
We implement the same **semantic element classification** directly from pdfminer.six
`LTTextBox` attributes — same element types, zero platform dependency.

### Classification heuristics

| Signal | Rule | Element type |
|--------|------|--------------|
| Short text + large font | `avg_font_size ≥ TITLE_FONT_THRESH` AND `len ≤ 120` | `Title` |
| Short text at page top (y ≥ 0.88) | Any short line at very top | `Header` |
| Short text at page bottom (y ≤ 0.06) | Any short line at very bottom | `Footer` |
| Starts with bullet/number | `^[-•*·◦▪\d]+[\.\)]\s` | `ListItem` |
| Looks like a table row | `≥ 2 tab-separated or spaced columns` | `Table` |
| Caption pattern | `^(Fig|Figure|Table|Chart)\b` (case-insensitive) | `FigureCaption` |
| Otherwise | Normal paragraph | `NarrativeText` |


In [ ]:
# ── Classifier thresholds ──
TITLE_FONT_THRESH = 13.0   # avg font size above this → Title candidate
LIST_RE   = re.compile(r'^[\-\•\*\·\◦\▪]|\d+[\.\)]\s')
CAPTION_RE = re.compile(r'^(fig|figure|table|chart|exhibit)\b', re.IGNORECASE)
TABLE_RE  = re.compile(r'\s{4,}|\t')   # multiple spaces or tab → table-like


def _avg_font_size(box) -> float:
    sizes = [c.size for c in box if isinstance(c, LTChar) and c.size > 0]
    return sum(sizes) / len(sizes) if sizes else 12.0


def classify_box(box, page_h: float) -> type:
    """Classify an LTTextBox into an element type class."""
    text   = box.get_text().strip()
    if not text:
        return type(None)

    y_norm = box.y0 / page_h

    # Page chrome first
    if y_norm >= 0.88 and len(text) < 100:
        return Header
    if y_norm <= 0.06 and len(text) < 100:
        return Footer

    avg_fs = _avg_font_size(box)

    # Title: large font, short text, not a list
    if avg_fs >= TITLE_FONT_THRESH and len(text) <= 120 and '\n' not in text.strip():
        return Title

    # ListItem: starts with bullet or number
    if LIST_RE.match(text):
        return ListItem

    # FigureCaption
    if CAPTION_RE.match(text):
        return FigureCaption

    # Table: row-like multi-column layout
    lines = [l for l in text.split('\n') if l.strip()]
    table_lines = sum(1 for l in lines if TABLE_RE.search(l))
    if lines and table_lines / len(lines) >= 0.5:
        return Table

    return NarrativeText


def partition_pdf_pdfminer(path: str) -> List:
    """
    Replaces unstructured.partition.pdf.partition_pdf().
    Returns a list of element objects (Title, NarrativeText, ListItem, Table,
    FigureCaption, Header, Footer) with .text and .metadata.page_number.
    """
    laparams = LAParams(line_margin=0.5, word_margin=0.1,
                        char_margin=2.0, boxes_flow=0.5)
    elements = []

    for page_num, page_layout in enumerate(
            extract_pages(path, laparams=laparams), start=1):
        page_h = page_layout.height or 1.0

        boxes = sorted(
            [b for b in page_layout if isinstance(b, LTTextBox)],
            key=lambda b: -b.y0)   # top-to-bottom reading order

        for box in boxes:
            text = box.get_text().strip()
            if not text:
                continue
            cls = classify_box(box, page_h)
            if cls is type(None):
                continue
            elements.append(cls(text, page_num))

    return elements


def element_type_name(el) -> str:
    return type(el).__name__


def build_hierarchy(elements) -> List[Dict]:
    sections = []
    current  = {'section_id': 'preamble', 'title': 'Preamble',
                 'element_type': 'Title', 'children': []}

    for el in elements:
        if isinstance(el, SKIP_TYPES):
            continue
        if not isinstance(el, KEEP_TYPES):
            continue
        text = el.text.strip() if el.text else ''
        if not text:
            continue
        page = el.metadata.page_number if el.metadata else None

        if isinstance(el, Title):
            if current['children']:
                sections.append(current)
            sid = f"sec_{len(sections):04d}"
            current = {'section_id': sid, 'title': text,
                       'element_type': 'Title', 'children': []}
        else:
            current['children'].append({
                'type': element_type_name(el), 'text': text, 'page': page})

    if current['children']:
        sections.append(current)
    return sections


def sections_to_chunks(sections: List[Dict]) -> Tuple[List[Dict], List[Dict]]:
    fine_chunks, coarse_chunks = [], []
    for sec in sections:
        sid, title, children = sec['section_id'], sec['title'], sec['children']
        for child in children:
            text = child['text']
            for k in range(0, max(len(text), 1), MAX_FINE_CHARS):
                sub = text[k:k + MAX_FINE_CHARS].strip()
                if sub:
                    fine_chunks.append({
                        'text': sub, 'section_id': sid,
                        'section_title': title,
                        'element_type': child['type'],
                        'page': child['page'], 'level': 'fine'})
        coarse_text = (title + '\n' + '\n'.join(c['text'] for c in children))
        coarse_text = coarse_text[:MAX_COARSE_CHARS].strip()
        if coarse_text:
            coarse_chunks.append({
                'text': coarse_text, 'section_id': sid,
                'section_title': title,
                'n_children': len(children),
                'page': children[0]['page'] if children else None,
                'level': 'coarse'})
    return fine_chunks, coarse_chunks


# ── Run extraction ──
t0       = time.time()
elements = partition_pdf_pdfminer(PDF_PATH)
elapsed  = (time.time() - t0) * 1000

type_counts = {}
for el in elements:
    n = element_type_name(el)
    type_counts[n] = type_counts.get(n, 0) + 1

print(f"partition_pdf (pdfminer.six classifier) : {elapsed:.0f}ms")
print(f"Total elements : {len(elements)}")
print(f"Element types  : {dict(sorted(type_counts.items(), key=lambda x: -x[1]))}")

sections                   = build_hierarchy(elements)
fine_chunks, coarse_chunks = sections_to_chunks(sections)

print(f"\nSections      : {len(sections)}")
print(f"Fine chunks   : {len(fine_chunks)}")
print(f"Coarse chunks : {len(coarse_chunks)}")
print(f"\nSample sections:")
for s in sections[:5]:
    print(f"  [{s['section_id']}] '{s['title'][:60]}' — {len(s['children'])} children")


## Step 6 — Connect & Index Both Levels

In [ ]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

all_chunks = fine_chunks + coarse_chunks
print(f"\nEmbedding {len(all_chunks)} total chunks "
      f"({len(fine_chunks)} fine + {len(coarse_chunks)} coarse)...")

t0   = time.time()
embs = embed_batch([c['text'] for c in all_chunks], label='[index]')

docs = [
    {
        'text'     : all_chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'level'         : all_chunks[i]['level'],
            'section_id'    : all_chunks[i]['section_id'],
            'section_title' : all_chunks[i]['section_title'],
            'element_type'  : all_chunks[i].get('element_type', 'Section'),
            'page'          : all_chunks[i].get('page'),
            'char_count'    : len(all_chunks[i]['text']),
            'source'        : 'climate.pdf',
            'pdf_lib'       : 'unstructured',
        }
    }
    for i in range(len(all_chunks))
]
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")

fine_count   = sum(1 for d in docs if d['metadata']['level'] == 'fine')
coarse_count = sum(1 for d in docs if d['metadata']['level'] == 'coarse')
print(f"  fine: {fine_count}  |  coarse: {coarse_count}")


## Step 7 — Recursive Retrieval

`recursive_retrieve()` queries only the fine-level vectors. For each hit:
- **score ≥ CONFIDENCE_THRESH** → keep the fine chunk as-is
- **score < CONFIDENCE_THRESH** → look up the parent coarse chunk by `section_id`

This gives precise snippets for high-confidence matches and fuller section
context for weaker matches — without any re-embedding or re-querying.


In [ ]:
def recursive_retrieve(question: str, top_k: int = TOP_K,
                        conf_thresh: float = CONFIDENCE_THRESH,
                        verbose: bool = False) -> List[Dict]:
    """
    Retrieve fine chunks, recurse to coarse for low-confidence hits.
    Returns list of passage dicts with 'text', 'level_used', 'score', 'section_title'.
    """
    qvec = embed_text(question)
    hits = vs.search(qvec, top_k=top_k, level_filter='fine')

    passages    = []
    seen_sids   = set()   # avoid duplicate coarse chunks for same section

    for h in hits:
        score = h['score']
        meta  = h['metadata']
        sid   = meta.get('section_id', '')

        if score >= conf_thresh:
            # High confidence → keep fine chunk
            passages.append({
                'text'          : h['text'],
                'score'         : score,
                'level_used'    : 'fine',
                'section_title' : meta.get('section_title', ''),
                'element_type'  : meta.get('element_type', ''),
                'section_id'    : sid,
            })
            if verbose:
                print(f"  FINE   score={score:.3f} [{meta.get('element_type','')}] "
                      f"'{h['text'][:60]}...'")
        else:
            # Low confidence → recurse up to parent coarse chunk
            if sid and sid not in seen_sids:
                coarse = vs.get_by_section_id(sid)
                if coarse:
                    seen_sids.add(sid)
                    passages.append({
                        'text'         : coarse['text'],
                        'score'        : score,
                        'level_used'   : 'coarse',
                        'section_title': coarse['metadata'].get('section_title', ''),
                        'element_type' : 'Section',
                        'section_id'   : sid,
                    })
                    if verbose:
                        print(f"  COARSE score={score:.3f} (recurse) "
                              f"'{coarse['text'][:60]}...'")
                    continue
            # Fallback: use fine even if coarse not found
            passages.append({
                'text'         : h['text'],
                'score'        : score,
                'level_used'   : 'fine_fallback',
                'section_title': meta.get('section_title', ''),
                'element_type' : meta.get('element_type', ''),
                'section_id'   : sid,
            })

    return passages


# Spot-test
test_passages = recursive_retrieve(
    "What methods are used for weather prediction?", verbose=True)
print(f"\nRetrieved {len(test_passages)} passages")
print(f"  fine: {sum(1 for p in test_passages if p['level_used']=='fine')}")
print(f"  coarse: {sum(1 for p in test_passages if p['level_used']=='coarse')}")


## Step 8 — Generator

In [ ]:
GEN_SYSTEM = (
    "You are a precise Q&A assistant. Answer using ONLY the provided context passages. "
    "Be concise but complete. If information is not present, say so briefly."
)

def generate_answer(question: str, passages: List[Dict]) -> str:
    context = '\n\n'.join(
        f'[{p["level_used"].upper()} | {p["section_title"]}]\n{p["text"]}'
        for i, p in enumerate(passages)
    )
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    return llm(prompt, system=GEN_SYSTEM).strip()


## Step 9 — Full Recursive RAG Pipeline

In [ ]:
def recursive_rag(question: str, top_k: int = TOP_K,
                  conf_thresh: float = CONFIDENCE_THRESH,
                  verbose: bool = True) -> Dict:
    t0 = time.time()

    passages = recursive_retrieve(question, top_k=top_k,
                                  conf_thresh=conf_thresh, verbose=verbose)
    answer   = generate_answer(question, passages)
    latency  = (time.time() - t0) * 1000

    fine_count   = sum(1 for p in passages if p['level_used'] == 'fine')
    coarse_count = sum(1 for p in passages if p['level_used'] == 'coarse')

    if verbose:
        print(f"\nAnswer:\n{answer}")
        print(f"\nLatency: {latency:.0f}ms  |  "
              f"fine={fine_count}  coarse={coarse_count}")
        print("-" * 70)

    return {
        'question'    : question,
        'answer'      : answer,
        'passages'    : passages,
        'n_fine'      : fine_count,
        'n_coarse'    : coarse_count,
        'latency_ms'  : latency,
        'conf_thresh' : conf_thresh,
    }


# Demo
result = recursive_rag("How do numerical weather prediction models work?")


## Step 10 — Multi-Question Demo

In [ ]:
test_questions = [
    # Specific → expect mostly fine chunks
    "What is ensemble forecasting?",
    # Broad → expect more coarse chunks (low confidence on any single element)
    "Describe the full range of observational inputs used in modern meteorology.",
    # Mixed
    "How do satellites contribute to weather forecasting?",
    # Table/list heavy — tests ListItem and Table element preservation
    "What are the main types of weather models and their characteristics?",
]

all_results = []
for q in test_questions:
    r = recursive_rag(q, verbose=False)
    all_results.append(r)

print(f"{'Question':<55}  {'Fine':>5}  {'Coarse':>6}  {'ms':>6}")
print("-" * 77)
for r in all_results:
    print(f"{r['question'][:54]:<55}  {r['n_fine']:>5}  {r['n_coarse']:>6}  "
          f"{r['latency_ms']:>6.0f}")

total_fine   = sum(r['n_fine']   for r in all_results)
total_coarse = sum(r['n_coarse'] for r in all_results)
print()
print(f"Totals — fine: {total_fine}  coarse: {total_coarse}  "
      f"coarse rate: {total_coarse/(total_fine+total_coarse)*100:.0f}%")


## Step 11 — Confidence Threshold Sweep

Show how the fine/coarse split changes across different confidence thresholds
for the same question.


In [ ]:
sweep_q      = "Explain the role of radar in weather observation and forecasting."
thresholds   = [0.5, 0.65, 0.75, 0.85, 0.95]

print(f"Threshold sweep for:\n  '{sweep_q}'\n")
print(f"{'Threshold':>10}  {'Fine':>5}  {'Coarse':>6}  {'Total chars':>11}")
print("-" * 38)

for thresh in thresholds:
    r = recursive_rag(sweep_q, conf_thresh=thresh, verbose=False)
    total_chars = sum(len(p['text']) for p in r['passages'])
    print(f"{thresh:>10.2f}  {r['n_fine']:>5}  {r['n_coarse']:>6}  {total_chars:>11}")

print()
print("Interpretation:")
print("  Low threshold  → almost all fine chunks (precise, short)")
print("  High threshold → more recursion to coarse (broader context, longer)")


## Step 12 — Summary

### What Recursive RAG does differently

| Dimension | Simple RAG | Recursive RAG |
|-----------|-----------|---------------|
| Index granularity | Single level | Fine + coarse (two levels) |
| Retrieval | Top-K flat | Top-K fine, then recurse up on low confidence |
| Context precision | Fixed chunk size | Adaptive: precise when confident, broad when not |
| Indexing cost | 1× embeddings | ~2× embeddings (fine + coarse overlap) |
| Latency | Low | Slightly higher (coarse lookups on low-confidence hits) |
| Best for | Uniform queries | Mixed queries: some precise, some needing context |

### unstructured element types used

| Type | RAG role |
|------|----------|
| `Title` | Section boundary → coarse chunk header |
| `NarrativeText` | Primary body → fine chunk |
| `ListItem` | Enumerated facts → fine chunk (kept intact) |
| `Table` | Structured data → fine chunk (kept whole) |
| `FigureCaption` | Annotation text → fine chunk |
| `Header` / `Footer` | Filtered out — page chrome, not content |

### PDF Framework Progression

| Notebook | Pattern | Framework | Key capability leveraged |
|----------|---------|-----------|--------------------------|
| 18 | Corrective RAG | **pdfplumber** | `bbox_density` as grader signal |
| 19 | Self RAG | **pymupdf** | Speed + font-size/bold → `is_heading` |
| 20 | Iterative RAG | **pdfminer.six** | `LTTextBox.y0` → layout zone tagging |
| 21 | Recursive RAG | **unstructured** | Semantic element types → two-level hierarchy |
| 22 | Graph RAG | **docling** | Table + relationship extraction |
| 23 | Speculative RAG | **marker** | Markdown-quality layout preservation |

> **Next: [22 — Graph RAG](../tier4_agentic/22_Graph_RAG.ipynb)** — `docling` table/relationship extraction feeds a lightweight entity-relationship graph; graph traversal augments vector retrieval.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {vs.count()} vectors indexed")
print(f"  fine: {len(fine_chunks)}  |  coarse: {len(coarse_chunks)}  |  sections: {len(sections)}")
print(f"PDF framework : pdfminer.six + heuristic classifier")
print(f"  (same element types as unstructured — Title/NarrativeText/ListItem/Table/FigureCaption)")
print(f"RAG pattern   : Recursive RAG (fine retrieval + coarse recursion on low confidence)")
print(f"Conf. threshold: {CONFIDENCE_THRESH}")
print()
print("Notebook 21 — Recursive RAG complete.")
